# Context for each alert

The validation notebook measured how far the alerts can be trusted as a whole. This notebook brings that judgement down to each single alert: what PRODES says about the ground it sits on, which PRODES years it overlaps, and how strongly the foundation model sees a change there.

The result, `outputs/alerts_enriched.geojson`, is what the assistant reads to answer questions about a specific alert. Computing it once here keeps the assistant fast and independent of the large PRODES file.

## 1. Settings

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray
from rasterio.enums import Resampling
from scipy.stats import percentileofscore

from utils import to_mask, share_in_polygons, land_status, mean_value_in_polygon

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")

PRODES_PATH = DATA_DIR / "prodes_amazonia_legal_v20260717.gpkg"
YEARS_COMPARED = (2019, 2023)     # same PRODES years as the validation notebook
STATUS_THRESHOLD = 0.5            # share of an alert needed to give it a land status

## 2. Alerts, grid and PRODES

The alerts are brought to the UTM grid of the NDVI change map, where every mask of the validation notebook was drawn. The PRODES layers are the same as there: clearings mapped in 2019 to 2023, and all the land PRODES considered cleared or not forest by 2018.

In [2]:
grid = rioxarray.open_rasterio(OUTPUT_DIR / "ndvi_change.tif").squeeze("band", drop=True)
alerts = gpd.read_file(OUTPUT_DIR / "deforestation_alerts.geojson").to_crs(grid.rio.crs)

read_bbox = grid.rio.transform_bounds("EPSG:4674")
yearly = gpd.read_file(PRODES_PATH, layer="yearly_deforestation", bbox=read_bbox)
prodes_recent = yearly[yearly["year"].between(*YEARS_COMPARED)].to_crs(grid.rio.crs)
prodes_earlier = yearly[yearly["year"] < YEARS_COMPARED[0]]
accumulated = gpd.read_file(PRODES_PATH, layer="accumulated_deforestation_2007", bbox=read_bbox)
no_forest = gpd.read_file(PRODES_PATH, layer="no_forest", bbox=read_bbox)
hydrography = gpd.read_file(PRODES_PATH, layer="hydrography", bbox=read_bbox)

not_forest_2018 = (to_mask(accumulated, grid) | to_mask(prodes_earlier, grid)
                   | to_mask(no_forest, grid) | to_mask(hydrography, grid))

print(len(alerts), "alerts,", len(prodes_recent), "PRODES clearings 2019-2023")

129 alerts, 30 PRODES clearings 2019-2023


## 3. What PRODES says under each alert

For every alert, the share of its area that PRODES mapped as cleared in 2019 to 2023, the share that was already cleared before 2019, and the rest, forest that PRODES still monitors. The three add up to one, since PRODES never records the same ground twice.

From these shares each alert gets a land status, the one covering at least half of it:

- **mapped by PRODES**: confirmed by the official map;
- **cleared before 2019**: land PRODES no longer monitors, so it can neither confirm nor reject the alert. Most likely a change of land use, such as pasture or crops, rather than new deforestation;
- **forest, not in PRODES**: a real disagreement, on land PRODES still watches;
- **mixed**: no single status covers half of the alert.

The PRODES years each alert overlaps are listed too, so the assistant can say when a confirmed clearing was mapped.

In [3]:
alerts["prodes_share"] = share_in_polygons(alerts, to_mask(prodes_recent, grid), grid)
alerts["cleared_before_2019_share"] = share_in_polygons(alerts, not_forest_2018, grid)
alerts["forest_share"] = (1 - alerts["prodes_share"] - alerts["cleared_before_2019_share"]).clip(lower=0)

hits = gpd.sjoin(alerts[["priority_rank", "geometry"]], prodes_recent[["year", "geometry"]],
                 predicate="intersects")
years = hits.groupby("priority_rank")["year"].apply(lambda y: sorted(set(int(v) for v in y)))
alerts["prodes_years"] = alerts["priority_rank"].map(years).apply(lambda y: y if isinstance(y, list) else [])

alerts["land_status"] = [land_status(p, c, STATUS_THRESHOLD)
                         for p, c in zip(alerts["prodes_share"], alerts["cleared_before_2019_share"])]

alerts[["priority_rank", "area_ha", "prodes_share", "cleared_before_2019_share",
        "forest_share", "prodes_years", "land_status"]].head(10).round(2)

,priority_rank,area_ha,prodes_share,cleared_before_2019_share,forest_share,prodes_years,land_status
0,1,176.99,0.01,0.98,0.01,[2022],cleared before 2019
1,2,163.47,0.00,0.99,0.01,[],cleared before 2019
2,3,103.38,0.94,0.06,0.01,"[2019, 2020, 2023]",mapped by PRODES
3,4,53.67,0.38,0.57,0.04,[2019],cleared before 2019
4,5,46.83,0.08,0.88,0.04,"[2020, 2023]",cleared before 2019
5,6,31.04,0.83,0.17,0.00,[2019],mapped by PRODES
6,7,28.49,0.00,0.84,0.16,[2019],cleared before 2019
7,8,22.75,0.00,1.00,0.00,[],cleared before 2019
8,9,21.75,0.00,1.00,0.00,[],cleared before 2019
9,10,20.47,0.55,0.35,0.11,"[2019, 2022]",mapped by PRODES


## 4. A check against the validation notebook

Summed over all alerts, area times share must give back the pixel-level numbers of the validation notebook: about 242 ha of alerts matching PRODES (the true positives) and about 754 ha on land already cleared. A different result would mean the shares are computed on a different grid or with different layers.

In [4]:
print(f"alert area matching PRODES 2019-2023: {(alerts['area_ha'] * alerts['prodes_share']).sum():.1f} ha")
print(f"alert area on land cleared before 2019: {(alerts['area_ha'] * alerts['cleared_before_2019_share']).sum():.1f} ha")
print(f"alert area on forest, not in PRODES: {(alerts['area_ha'] * alerts['forest_share']).sum():.1f} ha")

alert area matching PRODES 2019-2023: 242.1 ha
alert area on land cleared before 2019: 753.9 ha
alert area on forest, not in PRODES: 54.3 ha


## 5. What the foundation model sees under each alert

The Prithvi distance map has 480 m pixels. It is resampled to the 10 m grid with nearest neighbour, so every 10 m pixel takes the value of the patch it lies in, and the distance is averaged inside each alert. The percentile places that value among all 529 patches of the AOI.

This is a coarse context, not a verdict: a 480 m patch covers about 23 ha, so a small alert shares its value with everything around it.

In [5]:
prithvi = rioxarray.open_rasterio(OUTPUT_DIR / "prithvi_change.tif").squeeze("band", drop=True)
prithvi_10m = prithvi.rio.reproject_match(grid, resampling=Resampling.nearest)

alerts["prithvi_distance"] = alerts.geometry.apply(mean_value_in_polygon, args=(prithvi_10m, alerts.crs))
all_patches = prithvi.values.ravel()
alerts["prithvi_percentile"] = [percentileofscore(all_patches, d) for d in alerts["prithvi_distance"]]

alerts[["priority_rank", "area_ha", "land_status", "prithvi_distance", "prithvi_percentile"]].head(10).round(3)

,priority_rank,area_ha,land_status,prithvi_distance,prithvi_percentile
0,1,176.99,cleared before 2019,0.095,91.115
1,2,163.47,cleared before 2019,0.088,88.280
2,3,103.38,mapped by PRODES,0.196,99.244
3,4,53.67,cleared before 2019,0.129,97.732
4,5,46.83,cleared before 2019,0.095,91.115
5,6,31.04,mapped by PRODES,0.166,98.677
6,7,28.49,cleared before 2019,0.072,79.773
7,8,22.75,cleared before 2019,0.081,85.066
8,9,21.75,cleared before 2019,0.084,86.011
9,10,20.47,mapped by PRODES,0.107,93.762


## 6. Alerts by land status

How many alerts, and how much area, fall in each status, and how strongly the model sees a change on average in each group.

In [6]:
by_status = alerts.groupby("land_status").agg(
    alerts=("priority_rank", "count"),
    area_ha=("area_ha", "sum"),
    mean_prithvi_percentile=("prithvi_percentile", "mean"),
)
by_status["share_of_alert_area"] = by_status["area_ha"] / alerts["area_ha"].sum()
by_status.round(2)

,alerts,area_ha,mean_prithvi_percentile,share_of_alert_area
land_status,,,,
cleared before 2019,99,759.52,74.85,0.72
"forest, not in PRODES",17,35.86,60.84,0.03
mapped by PRODES,13,254.93,91.45,0.24


## 7. Saving

The enriched alerts are saved in EPSG:4326, like the original ones, with all the columns the assistant needs. The PRODES years are stored as text ("2020, 2021"), since GeoJSON has no column type for lists.

In [7]:
enriched = alerts.copy()
share_columns = ["prodes_share", "cleared_before_2019_share", "forest_share"]
enriched[share_columns] = enriched[share_columns].round(3)
enriched["prithvi_distance"] = enriched["prithvi_distance"].round(4)
enriched["prithvi_percentile"] = enriched["prithvi_percentile"].round(1)
enriched["prodes_years"] = enriched["prodes_years"].apply(lambda years: ", ".join(map(str, years)))

enriched.to_crs(4326).to_file(OUTPUT_DIR / "alerts_enriched.geojson", driver="GeoJSON")
print(len(enriched), "alerts saved with columns:", [c for c in enriched.columns if c != "geometry"])

129 alerts saved with columns: ['area_m2', 'mean_bsi_change', 'mean_ndvi_change', 'area_ha', 'severity', 'centroid_lon', 'centroid_lat', 'priority_rank', 'prodes_share', 'cleared_before_2019_share', 'forest_share', 'prodes_years', 'land_status', 'prithvi_distance', 'prithvi_percentile']
